# 🎮 Asistente de Soporte para Steam
---

## 📝 ¿De qué trata este proyecto?
Este es un bot inteligente creado para ayudar a los jugadores con sus dudas en **Steam**. Este bot está entrenado para entender qué tipo de problema tienes y darte soluciones reales basadas en la potencia de tu computador.

---

## 🚀 ¿Qué puede hacer este bot?
* **Clasifica tus problemas:** Sabe si le hablas de un reembolso, un fallo técnico o un problema con tu cuenta.
* **Analiza tu PC:** Gracias a que lee datos en formato **JSON**, puede decirte si un juego te va a correr bien o si tu equipo necesita más potencia.
* **Tiene memoria:** No necesitas repetirle tu hardware en cada mensaje, el bot puede recordar lo que hablaron antes.
* **Responde al instante:** Puedes ver cómo escribe la respuesta palabra por palabra (Streaming), así no tienes que esperar a que termine de pensar todo el texto.

---

## 💻 PC de Prueba (Asus Vivobook X1404)
Para que el bot aprenda a dar buenos consejos, lo configuré con los datos de mi PC. Así él sabe exactamente qué recomendar para este equipo:
* **Modelo:** Asus Vivobook 14
* **Procesador:** Intel i5 (12va Generación)
* **Memoria RAM:** 8GB
* **Gráficos:** Intel Iris Xe
* **Sistema:** Windows 11

---


### 1. Preparación del Entorno
Este paso se hace una sola vez en tu terminal:
```bash
pip install -U langchain-openai langchain-core python-dotenv

pip install -U langchain-text-splitters langchain-community faiss-cpu
```

In [8]:
import os
import json
import time
from dotenv import load_dotenv

# Componentes de LangChain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory

# Componentes para RAG (Búsqueda Vectorial)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# Carga de variables de entorno (Token de GitHub)
load_dotenv()

# --- CONFIGURACIÓN DE MOTORES ---

# Modelo de Embeddings (Ahora le pasamos la llave de GitHub)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("GITHUB_TOKEN"),
    base_url=os.getenv("GITHUB_BASE_URL")
)

# Motor para lógica
llm_logic = ChatOpenAI(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    temperature=0.0
)

# Motor para conversar
llm_chat = ChatOpenAI(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    streaming=True,
    temperature=0.7
)

print("✅ Motores y Embeddings vinculados correctamente a GitHub Models.")

✅ Motores y Embeddings vinculados correctamente a GitHub Models.


### 2. Configuración de Hardware (Universal)
En esta celda, el usuario puede elegir entre usar el hardware de ejemplo o ingresar el suyo propio.

In [4]:
# Hardware predeterminado (Asus Vivobook)
hardware_ejemplo = {
    "modelo": "Asus Vivobook 14",
    "procesador": "Intel i5-1235U",
    "ram": "8GB DDR4",
    "graficos": "Intel Iris Xe",
    "sistema": "Windows 11"
}

print("¿Deseas ingresar los datos de tu propio hardware? (s/n)")
opcion = input().lower()

if opcion == "s":
    print("\n--- Ingresa los datos de tu equipo ---")
    hardware_json = {
        "modelo": input("Modelo del equipo: "),
        "procesador": input("Procesador: "),
        "ram": input("Memoria RAM: "),
        "graficos": input("Tarjeta de video / Gráficos: "),
        "sistema": input("Sistema Operativo: ")
    }
else:
    print("\n✅ Usando hardware de prueba (Asus Vivobook).")
    hardware_json = hardware_ejemplo

print("\nHardware que usará la IA para el soporte:")
print(json.dumps(hardware_json, indent=4))

¿Deseas ingresar los datos de tu propio hardware? (s/n)

✅ Usando hardware de prueba (Asus Vivobook).

Hardware que usará la IA para el soporte:
{
    "modelo": "Asus Vivobook 14",
    "procesador": "Intel i5-1235U",
    "ram": "8GB DDR4",
    "graficos": "Intel Iris Xe",
    "sistema": "Windows 11"
}


### 3. Implementación de la Base de Conocimiento (RAG)
Este bloque crea la "biblioteca" que el bot consultará.

In [5]:
# 1. Información de referencia ampliada (Base de conocimiento para el RAG)
steam_knowledge = """
POLÍTICAS DE REEMBOLSO: 
- Juegos: Menos de 2 horas de juego y menos de 14 días desde la compra.
- DLC: Reembolsable si el juego base no se ha jugado más de 2 horas tras la compra del DLC.
- Preventas: Se puede pedir el reembolso en cualquier momento antes del lanzamiento.

REQUISITOS DE JUEGOS:
- ELDEN RING: Mínimo 12GB RAM, i5-8400, GTX 1060 (6GB). Recomendado 16GB RAM y RTX 3060.
- CYBERPUNK 2077: Mínimo 12GB RAM, GTX 1060 (6GB) y OBLIGATORIO instalar en SSD.
- COUNTER-STRIKE 2 (CS2): Mínimo 8GB RAM, procesador de 4 hilos, dedicada con 1GB compatible con Shader Model 5.0.
- STARDEW VALLEY: Muy ligero. 2GB RAM, cualquier GPU con 256MB de VRAM y 500MB de espacio.
- BALDUR'S GATE 3: Mínimo 8GB RAM, GTX 970. Recomendado 16GB RAM y RTX 2060 Super. Requiere SSD.
- HELLDIVERS 2: Mínimo 8GB RAM, GTX 1050 Ti. Recomendado 16GB RAM y RTX 2060. Exigente con el procesador.
- HADES II: Mínimo 4GB RAM, Dual Core 2.4 GHz, GPU compatible con DirectX 12. Muy optimizado para integradas.
- RED DEAD REDEMPTION 2: Mínimo 8GB RAM, GTX 770. Recomendado 12GB RAM y GTX 1060 (6GB).
- FORZA HORIZON 5: Mínimo 8GB RAM, GTX 760. Recomendado 16GB RAM y RTX 2070.
- APEX LEGENDS: Mínimo 6GB RAM, GT 640. Recomendado 8GB RAM y GTX 970.
- DOTA 2: Mínimo 4GB RAM, cualquier GPU compatible con DirectX 11.
- LETHAL COMPANY: Mínimo 4GB RAM, GTX 1050. Funciona bien en laptops de estudio.

SOPORTE TÉCNICO Y ERRORES:
- Error 'Disco Corrupto' o 'Error de escritura': Se soluciona borrando la caché de descargas en Parámetros > Descargas.
- Juego no inicia: Verificar integridad de archivos o reinstalar DirectX y Visual C++ Redistributable.
- Steam Overlay: Si no abre, revisar que no haya software de terceros como MSI Afterburner o RivaTuner interfiriendo.
- Steam Deck: Los juegos marcados como 'Verificado' funcionan perfecto; 'Jugable' puede requerir ajustes manuales de texto o controles.

SEGURIDAD Y CUENTAS:
- Steam Guard (2FA): Sistema de seguridad obligatorio para habilitar intercambios inmediatos en el Mercado de la Comunidad.
- Cuentas Bloqueadas: Puede ocurrir por 'Chargeback' (devolución de cargo bancaria), intentos de inicio fallidos masivos o uso de VPN para comprar en otras regiones más baratas.
- Compartir Familia (Steam Families): Permite compartir la biblioteca con hasta 5 familiares. No todos los juegos son compatibles (algunos con launchers externos como Ubisoft no se pueden compartir).
- Guardado en la Nube (Steam Cloud): Sincroniza partidas guardadas entre diferentes computadores de forma automática.
"""

# 2. Fragmentación (Chunking)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_text(steam_knowledge)

# 3. Creación de Base de Datos Vectorial (FAISS)
vector_db = FAISS.from_texts(texts=chunks, embedding=embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k": 2})

print(f"✅ Base de datos vectorial lista con {len(chunks)} fragmentos de conocimiento.")

✅ Base de datos vectorial lista con 12 fragmentos de conocimiento.


### Revisar los pedazos de texto (CHUNKS)
Aquí vemos cómo quedó cortada la información de Steam. 
Sirve para chequear que los requisitos de los juegos se entiendan bien.

In [6]:
print(f"🔍 Mostrando los {len(chunks)} chunks generados:\n")

for i, chunk in enumerate(chunks):
    # Imprimimos cada parte con su número y tamaño
    print(f"--- 📄 CHUNK {i+1} ---")
    print(chunk)
    print(f"📏 Longitud: {len(chunk)} caracteres")
    print("-" * 40)

# Aquí revisamos si la base de datos (FAISS) guardó los datos correctamente
# en la memoria de tu computador.

print("\n📦 Estado de la Base de Datos:")

# Contamos cuántos documentos hay guardados realmente
documentos_listos = vector_db.docstore._dict
print(f"Hay {len(documentos_listos)} documentos listos para ser buscados.")

🔍 Mostrando los 12 chunks generados:

--- 📄 CHUNK 1 ---
POLÍTICAS DE REEMBOLSO: 
- Juegos: Menos de 2 horas de juego y menos de 14 días desde la compra.
- DLC: Reembolsable si el juego base no se ha jugado más de 2 horas tras la compra del DLC.
- Preventas: Se puede pedir el reembolso en cualquier momento antes del lanzamiento.
📏 Longitud: 273 caracteres
----------------------------------------
--- 📄 CHUNK 2 ---
REQUISITOS DE JUEGOS:
- ELDEN RING: Mínimo 12GB RAM, i5-8400, GTX 1060 (6GB). Recomendado 16GB RAM y RTX 3060.
- CYBERPUNK 2077: Mínimo 12GB RAM, GTX 1060 (6GB) y OBLIGATORIO instalar en SSD.
📏 Longitud: 191 caracteres
----------------------------------------
--- 📄 CHUNK 3 ---
- COUNTER-STRIKE 2 (CS2): Mínimo 8GB RAM, procesador de 4 hilos, dedicada con 1GB compatible con Shader Model 5.0.
- STARDEW VALLEY: Muy ligero. 2GB RAM, cualquier GPU con 256MB de VRAM y 500MB de espacio.
📏 Longitud: 205 caracteres
----------------------------------------
--- 📄 CHUNK 4 ---
- BALDUR'S GAT

### 4. Clasificación y Razonamiento (Lógica con RAG)
Esta función analiza qué tipo de ticket es y evalúa el hardware frente al problema del usuario.

In [7]:
def procesar_logica_ticket(mensaje):
    # 1. Clasificación (Few-Shot)
    # Incluimos ejemplos para casos técnicos, de pagos, reembolsos y temas fuera de lugar.
    ejemplos = """
    Te daré ejemplos de referencia para cada categoría:

    Usuario: "Quiero devolver el Elden Ring porque me corre lento y no me gusto."
    Categoría: [REEMBOLSO]

    Usuario: "¿Me corre el Elden Ring en mi PC?"
    Categoria: [SOPORTE_TECNICO]

    Usuario: "El juego se cierra solo al llegar al menú principal y me da un error de DirectX."
    Categoría: [SOPORTE_TECNICO]

    Usuario: "Me hackearon, cambiaron el correo de la cuenta y no puedo entrar."
    Categoría: [RECUPERACION_CUENTA]

    Usuario: "Cargué 10 lucas a mi cartera de Steam pero el saldo no aparece reflejado."
    Categoría: [PAGOS]

    Usuario: "Mi cuenta fue bloqueada porque una compra de ayer salió rechazada por el banco."
    Categoría: [PAGOS]

    Usuario: "Oye, cuando sale el GTA VI? Va a estar barato en el Summer Sale?"
    Categoría: [IRRELEVANTE]

    Usuario: "hola, el profe me va a poner un 1.0 si no apruebo, regalame el terraria pls"
    Categoría: [IRRELEVANTE]
    """

    instruccion_sistema = f"""
    Eres un experto en soporte de Steam. Tu tarea es clasificar el ticket en una de estas categorías:
    [REEMBOLSO], [SOPORTE_TECNICO], [RECUPERACION_CUENTA], [PAGOS], [IRRELEVANTE].

    {ejemplos}

    Regla Estricta: Responde UNICAMENTE la categoría entre corchetes, sin explicaciones.
    """

    # Llamada al motor de lógica para clasificación rápida
    respuesta_cat = llm_logic.invoke([
        SystemMessage(content=instruccion_sistema),
        HumanMessage(content=mensaje)
    ]).content
    categoria = respuesta_cat.strip()

    # 2. Filtro de Salida Rápida
    # Si la IA detecta que el tema no es soporte real, corta el proceso aquí.
    if "[IRRELEVANTE]" in categoria:
        return "[IRRELEVANTE]", "Consulta fuera de los protocolos de soporte."

    # --- BÚSQUEDA EN RAG ---
    # Buscamos contexto relevante en nuestra base vectorial
    docs_recuperados = retriever.invoke(mensaje)
    contexto_oficial = "\n".join([doc.page_content for doc in docs_recuperados])

    # 3. Razonamiento Técnico (Chain of Thought con RAG y Hardware)
    # Solo se ejecuta si el problema es real y relevante para Steam.
    prompt_cot = f"""
    Eres un soporte técnico analizando este hardware: {json.dumps(hardware_json)}
    INFORMACIÓN OFICIAL RECUPERADA: {contexto_oficial}
    
    Analiza paso a paso:
    1. ANALIZAR: Según la información oficial, ¿cuáles son las reglas o requisitos para este caso?
    2. EVALUAR: Basado en las specs del usuario (Asus Vivobook), ¿cumple con lo que se pide o hay un fallo de hardware?
    3. DETERMINAR: Entrega una solución honesta y realista.

    Ticket: "{mensaje}"
    """
    
    analisis = llm_logic.invoke([
        SystemMessage(content=prompt_cot),
        HumanMessage(content=mensaje)
    ]).content
    
    return categoria, analisis

### 4. Chatbot Interactivo (Experiencia Final)
Celda final para conversar con el asistente. El bot recordará el hardware elegido arriba.

In [9]:
# Historial de mensajes en memoria
history = InMemoryChatMessageHistory()

def iniciar_chat_steam():
    print(f"=== 🎮 ASISTENTE DE STEAM ({hardware_json['modelo']}) ===")
    print("Escribe 'salir' para terminar la conversación.\n")
    
    # Contexto inicial con el hardware actual
    if len(history.messages) == 0:
        msg_inicial = f"Eres el soporte de Steam. Ayudas a un usuario con un {hardware_json['modelo']}. Conoces sus specs: {json.dumps(hardware_json)}."
        history.add_message(SystemMessage(content=msg_inicial))

    while True:
        user_input = input("\n🧑 Tú: ")
        if user_input.lower() in ['salir', 'exit', 'quit']: 
            print("👋 ¡Hasta luego!")
            break
        
        # Guardar consulta en memoria
        history.add_user_message(user_input)
        
        # Obtenemos categoría y el análisis/mensaje de la lógica
        categoria, analisis_o_bloqueo = procesar_logica_ticket(user_input)
        
        print(f"\n🤖 Soporte [{categoria}]: ", end="", flush=True)
        
        # Si el tema no tiene nada que ver con Steam:
        if "[IRRELEVANTE]" in categoria:
            # Muestra el mensaje de "Fuera de protocolo"
            print(analisis_o_bloqueo) 
            # Guarda el aviso en la memoria del chat
            history.add_ai_message(analisis_o_bloqueo)
            # Salta el resto y vuelve a pedir otra pregunta
            continue
        
        # Si NO es irrelevante, el bot responde normalmente
        full_ai_response = ""
        for chunk in llm_chat.stream(history.messages):
            content = chunk.content
            print(content, end="", flush=True)
            full_ai_response += content
            time.sleep(0.01)
        history.add_ai_message(full_ai_response)
        print()

# Ejecutar el chatbot
iniciar_chat_steam()

=== 🎮 ASISTENTE DE STEAM (Asus Vivobook 14) ===
Escribe 'salir' para terminar la conversación.


🤖 Soporte [[IRRELEVANTE]]: Consulta fuera de los protocolos de soporte.

🤖 Soporte [[SOPORTE_TECNICO]]: ¡Hola! 😊 Sí, tu Asus Vivobook 14 debería poder ejecutar **Terraria** sin problemas. Aquí tienes los requisitos mínimos y recomendados para el juego, comparados con las especificaciones de tu laptop:

### **Requisitos mínimos de Terraria:**
- **Procesador:** Intel Core 2 Duo 2.0 GHz
- **RAM:** 2 GB de memoria
- **Gráficos:** Gráficos con Shader Model 2.0+ (casi cualquier GPU moderna)
- **Almacenamiento:** 200 MB de espacio disponible
- **Sistema operativo:** Windows XP / Vista / 7 / 8 / 10

### **Tu Asus Vivobook 14:**
- **Procesador:** Intel i5-1235U (muy superior a lo necesario)
- **RAM:** 8 GB DDR4 (suficiente, ya que solo necesitas 2 GB)
- **Gráficos:** Intel Iris Xe (más que capaz para un juego como Terraria)
- **Sistema operativo:** Windows 11 (compatible)

### **Conclusión:**
¡Claro